## Benchmarking VectoreRAG + GraphRAG Recommnedation System##

In [ ]:
import json
from collections import defaultdict

# Load tool and workflow
with open("../../utilities/tools_metadata_downloader/data/galaxy_instance_tools_2025-12-04_23-58-00.json", "r") as f:
    tools = json.load(f)

with open("../../utilities/workflow_downloader/data/galaxy_iwc_workflows_20251205_162934.json", "r") as f:
    workflows = json.load(f)

print(f"Loaded {len(tools)} tools and {len(workflows)} workflows.")

In [ ]:
import os, sys
import sys
import os

# Set project root
project_root = os.path.abspath("../../")  # go up two levels from graphRAG
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(sys.path)  # confirm it now includes project root


# Now imports should work
from agents.graphRAG.pipeline.tool_retrieval_pipeline import ToolRetrievalPipeline
from agents.graphRAG.pipeline.workflow_retrival_pipeline import WorkflowRetrievalPipeline
from agents.ingestion.Load.neo4j_client import Neo4jClient
from agents.scripts.query_embedding import QueryEmbeddingService

# Define project root (adjust if notebook is inside subfolder)
project_root = os.path.abspath("../../")  # go two levels up from graphRAG
config_path = os.path.join(project_root, "agents/graphRAG/config/graph_db_config.yml")

# Initialize Neo4j client
neo_client = Neo4jClient(config_path=config_path)
tool_pipeline = ToolRetrievalPipeline(neo_client)
workflow_pipeline = WorkflowRetrievalPipeline(neo_client)
embedder=QueryEmbeddingService()

## tool benchmarking

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def compute_tool_recall_with_embedding(
    tool_pipeline,
    queries,
    ground_truth,
    embedder,
    k=5,
    threshold=0.8
):
    """
    Compute recall@k for tools using embedding similarity.

    Args:
        tool_pipeline: ToolRetrievalPipeline
        queries: list[str]
        ground_truth: dict[str, list[str]]
        embedder: QueryEmbeddingService
        k: int, top-k retrieved tools to consider
        threshold: float, cosine similarity threshold to count as match

    Returns:
        recall_scores: list[int]
        avg_recall: float
    """
    recall_scores = []
    total_recall = 0

    for query in queries:
        results = tool_pipeline.retrieve_tools(query, top_k=k)

        # Extract tool names safely
        retrieved_items = []
        for ctx in results[:k]:
            tool = ctx.get("tool", {})
            name = tool.get("tool_name") or tool.get("name") or tool.get("tool_id")
            if name:
                retrieved_items.append(name)

        retrieved_items = list(set(retrieved_items))
        expected_items = ground_truth.get(query, [])

        if not expected_items or not retrieved_items:
            recall = 0
        else:
            # Compute embeddings
            expected_embeds = embedder.embed_query(expected_items)
            retrieved_embeds = embedder.embed_query(retrieved_items)

            sim_matrix = cosine_similarity(expected_embeds, retrieved_embeds)
            max_sims = sim_matrix.max(axis=1)

            recall = 1 if any(s >= threshold for s in max_sims) else 0

        recall_scores.append(recall)
        total_recall += recall

        print(f"\nQuery: {query}")
        print(f"Expected: {expected_items}")
        print(f"Top-{k} retrieved: {retrieved_items}")
        print(f"Recall@{k} : {recall}")

    avg_recall = total_recall / len(queries)
    print(f"\nAverage Recall@{k} : {avg_recall:.2f}")

    return recall_scores, avg_recall

In [ ]:
with open("benchmarks/tool_test_queries.json", "r") as f:
    dataset = json.load(f)

queries = dataset["queries"]
ground_truth = dataset["ground_truth"]

recall_scores, avg_recall = compute_tool_recall_with_embedding(
    tool_pipeline=tool_pipeline,
    queries=queries,
    ground_truth=ground_truth,
    embedder=embedder,
    k=5,
    threshold=0.8
)

## workflow benchmarking

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def compute_workflow_recall_with_embedding(
    workflow_pipeline,
    queries,
    ground_truth,
    embedder,
    k=5,
    threshold=0.8
):
    """
    Compute recall@k using embedding similarity between retrieved and expected workflows.

    Args:
        workflow_pipeline: WorkflowRetrievalPipeline
        queries: list[str]
        ground_truth: dict[str, list[str]]
        embedder: QueryEmbeddingService
        k: int
        threshold: float, cosine similarity threshold to count a match

    Returns:
        recall_scores: list[int]
        avg_recall: float
    """
    recall_scores = []
    total_recall = 0

    for query in queries:
        results = workflow_pipeline.retrieve_workflows(query, top_k=k)

        retrieved_names = []
        for ctx in results[:k]:
            wf = ctx.get("workflow", {})
            name = wf.get("workflow_name") or wf.get("name") or wf.get("workflow_id")
            if name:
                retrieved_names.append(name)

        retrieved_names = list(set(retrieved_names))
        expected_names = ground_truth.get(query, [])

        # Get embeddings for both expected and retrieved
        if not expected_names or not retrieved_names:
            recall = 0
        else:
            expected_embeds = embedder.embed_query(expected_names)
            retrieved_embeds = embedder.embed_query(retrieved_names)
            sim_matrix = cosine_similarity(expected_embeds, retrieved_embeds)
            max_sims = sim_matrix.max(axis=1)
            recall = 1 if any(s >= threshold for s in max_sims) else 0

        recall_scores.append(recall)
        total_recall += recall

        print(f"\nQuery: {query}")
        print(f"Expected: {expected_names}")
        print(f"Top-{k} retrieved: {retrieved_names}")
        print(f"Recall@{k}: {recall}")

    avg_recall = total_recall / len(queries)
    print(f"\nAverage Recall@{k}: {avg_recall:.2f}")

    return recall_scores, avg_recall

In [ ]:
with open("benchmarks/workflow_test_queries.json", "r") as f:
    dataset = json.load(f)

queries = dataset["queries"]
ground_truth = dataset["ground_truth"]


recall_scores, avg_recall = compute_workflow_recall_with_embedding(
    workflow_pipeline=workflow_pipeline,
    queries=queries,
    ground_truth=ground_truth,
    embedder=embedder,  
    k=5,
    threshold=0.8  
)